# Step 3 — Visual Evaluation and Integration Test

Loads the saved model and evaluates it on the held-out test set.
Also runs live predictions on raw frames for visual inspection.

**Inputs:**
- `models/svm_line_follower.pkl`
- `data/X_features.npy`, `data/y_labels.npy`
- Raw JPEG frames from `extracted_images/`

## 1. Imports and paths

In [ ]:
import pickle
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

REPO_ROOT  = Path(r"C:/Users/danie/Desktop/Autonomous_Team4/GitBranch_Simulation")
DATA_DIR   = REPO_ROOT / "Autonomous_Systems_A_Lab_Group_4" / "data"
MODELS_DIR = REPO_ROOT / "Autonomous_Systems_A_Lab_Group_4" / "models"
IMAGES_DIR = REPO_ROOT / "Autonomous_Systems_A_Lab_Group_4" / "extracted_images"

CLASS_NAMES = np.array(["LEFT", "STRAIGHT", "RIGHT"])
DEAD_BAND   = 0.10

## 2. Load model and dataset

In [ ]:
with open(str(MODELS_DIR / "svm_line_follower.pkl"), "rb") as f:
    model = pickle.load(f)

steer_map = np.array([-1.0, 0.0, 1.0], dtype=np.float32)
print("Model loaded:", model)

X = np.load(str(DATA_DIR / "X_features.npy"))
y = np.load(str(DATA_DIR / "y_labels.npy"))

assert X.shape[1] == 5, f"Expected 5 features, got {X.shape[1]} — re-run notebooks 01 and 02"
print(f"Dataset: X={X.shape}, y={y.shape}")
print("Class distribution:")
for i, name in enumerate(CLASS_NAMES):
    n = (y == i).sum()
    print(f"  {name:10s}: {n:5d}  ({100*n/len(y):.1f}%)")

## 3. Metrics on held-out test set

In [ ]:
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

y_pred = model.predict(X_test)

print("Test accuracy :", accuracy_score(y_test, y_pred))
print("Test macro F1 :", round(f1_score(y_test, y_pred, average="macro"), 4))
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=CLASS_NAMES,
).plot()
plt.title("Tuned SVM — Test Set (5 features)")
plt.show()

## 4. Feature pipeline (copied from notebook 01)

In [ ]:
LOWER_GREEN      = np.array([40,  40,  40])
UPPER_GREEN      = np.array([90, 255, 255])
MIN_CONTOUR_AREA = 100


def extract_features(img_bgr: np.ndarray):
    rows, cols = img_bgr.shape[:2]

    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    contour = max(contours, key=cv2.contourArea)
    area    = cv2.contourArea(contour)
    if area < MIN_CONTOUR_AREA:
        return None, None

    [vx, vy, x, y] = cv2.fitLine(contour, cv2.DIST_L2, 0, 0.01, 0.01)
    center_row = rows // 2
    if abs(float(vy)) > 0.01:
        line_x = float(x) + (center_row - float(y)) * (float(vx) / float(vy))
    else:
        M      = cv2.moments(contour)
        line_x = M["m10"] / M["m00"] if M["m00"] > 0 else cols / 2
    offset = float(np.clip((line_x - cols / 2.0) / (cols / 2.0), -1.0, 1.0))

    angle      = float(np.degrees(np.arctan2(float(vy), float(vx))))
    angle_norm = float(np.clip(angle / 90.0, -1.0, 1.0))
    area_norm  = float(np.clip(area / (rows * cols), 0.0, 1.0))

    _, _, bw, bh = cv2.boundingRect(contour)
    aspect_ratio = float(np.clip(bw / (bh + 1e-6), 0.0, 1.0))

    hull      = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity  = float(np.clip(area / (hull_area + 1e-6), 0.0, 1.0))

    feat = np.array([offset, angle_norm, area_norm, aspect_ratio, solidity], dtype=np.float32)
    return feat, offset


def offset_to_label(offset: float) -> int:
    if offset < -DEAD_BAND: return 0
    if offset >  DEAD_BAND: return 2
    return 1


print("Feature helpers defined.")

## 5. Visual grid — 6 sample frames per class

- Green border = correct prediction
- Red border = wrong prediction
- Grey border = no line detected

In [ ]:
image_paths = sorted(
    list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.jpeg")) + list(IMAGES_DIR.glob("*.JPG"))
)

sample_frames, sample_offsets = [], []
for img_path in image_paths[::30]:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    _, offset = extract_features(img)
    if offset is None:
        continue
    sample_frames.append(img)
    sample_offsets.append(offset)

print(f"Loaded {len(sample_frames)} sample frames")

rng           = np.random.default_rng(RANDOM_STATE)
sample_labels = np.array([offset_to_label(o) for o in sample_offsets])
selected      = []
for cls in range(3):
    indices = np.where(sample_labels == cls)[0]
    chosen  = rng.choice(indices, size=min(6, len(indices)), replace=False)
    selected.extend(chosen.tolist())

fig, axes = plt.subplots(3, 6, figsize=(14, 7))

for ax, idx in zip(axes.ravel(), selected):
    img    = sample_frames[idx]
    offset = sample_offsets[idx]
    true   = offset_to_label(offset)

    feat, _ = extract_features(img)
    vis     = cv2.resize(img, (320, 180)).copy()

    if feat is None:
        pred         = -1
        label_text   = "NO LINE"
        border_color = (128, 128, 128)
    else:
        pred         = model.predict(feat.reshape(1, -1))[0]
        label_text   = f"pred: {CLASS_NAMES[pred]}"
        border_color = (0, 200, 0) if pred == true else (0, 0, 220)

    vis = cv2.copyMakeBorder(vis, 4, 4, 4, 4, cv2.BORDER_CONSTANT, value=border_color)
    cv2.putText(vis, label_text,
                (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(vis, f"true: {CLASS_NAMES[true]} (off={offset:+.2f})",
                (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 220), 1, cv2.LINE_AA)

    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis("off")

for row, cls_name in enumerate(CLASS_NAMES):
    axes[row, 0].set_ylabel(cls_name, fontsize=11, rotation=0, labelpad=50, va="center")

plt.suptitle("6 examples per class  |  green=correct  red=wrong  grey=no line", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Noise robustness test

In [ ]:
noise_levels = [0.0, 0.05, 0.10, 0.20, 0.40]
rng2         = np.random.default_rng(RANDOM_STATE)
accuracies, f1_scores = [], []

for nl in noise_levels:
    X_noisy = X_test + rng2.normal(0, nl, X_test.shape)
    y_noisy = model.predict(X_noisy)
    acc     = round(accuracy_score(y_test, y_noisy), 4)
    f1      = round(f1_score(y_test, y_noisy, average="macro"), 4)
    accuracies.append(acc)
    f1_scores.append(f1)
    print(f"  noise_std={nl:.2f}  accuracy={acc:.4f}  macro_f1={f1:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(noise_levels, accuracies, marker="o", label="accuracy")
plt.plot(noise_levels, f1_scores,  marker="o", label="macro F1")
plt.xlabel("Feature noise std")
plt.ylabel("Score")
plt.title("Robustness to feature noise")
plt.legend()
plt.grid(True)
plt.show()

## 7. Summary

| | |
|---|---|
| Model file | `models/svm_line_follower.pkl` |
| SVM input | 5-element float32 vector: `[offset, angle_norm, area_norm, aspect_ratio, solidity]` |
| SVM output | Integer class: 0=LEFT, 1=STRAIGHT, 2=RIGHT |
| Steer map | `[-1.0, 0.0, +1.0]` |
| Prediction time | ~0.5ms (vs ~10ms with 4098 features) |

### Integration snippet for `my_line_follower.py`

```python
cls   = int(self._model.predict(feat.reshape(1, -1))[0])  # feat = 5-element vector
target = [-1.0, 0.0, 1.0][cls]                            # SVM decides direction
# PID smooths toward target
```